# Assignment 1: Weather Agent

This project uses the Gemini SDK to fetch live weather data for multiple cities through the OpenWeather API. The agent checks each location in order, collects the temperature for each one, and then calculates the average in Celsius.


## How It Works

**The system follows this workflow:**   
  
<br>

<div align="center">


User Request      
    ↓  
Gemini Agent   
    ↓  
Weather Tool   
    ↓  
OpenWeather API  
    ↓  
Location 1  
    ↓  
Location 2    
    ↓  
Location 3  
    ↓  
Python calculates average temperature  
    ↓  
Final Weather Report


</div>


## Locations

The three locations used for this assignment are:

1. Kathmandu, Nepal
2. London, United Kingdom
3. Tokyo, Japan

## 1. Setup and Dependencies

Install the Python libraries required for the Weather Agent.

In [5]:
!pip install -q google-genai requests

In [6]:
import os
import requests

from google import genai
from google.genai import types

print("Libraries imported successfully!")

Libraries imported successfully!


## 2. API Configuration

The weather agent needs two API keys:

- Gemini API key for the model
- OpenWeather API key for live weather data

These keys are stored in Google Colab Secrets so they are not exposed in the notebook itself.

```python
from google.colab import userdata

gemini_key = userdata.get('GEMINI_API_KEY')
openweather_key = userdata.get('OPENWEATHER_API_KEY')
```


In [7]:
from google.colab import userdata

GEMINI_API_KEY = userdata.get("GEMINI_API_KEY")
OPENWEATHER_API_KEY = userdata.get("OPENWEATHER_API_KEY")

print("Gemini API key loaded:", bool(GEMINI_API_KEY))
print("OpenWeather API key loaded:", bool(OPENWEATHER_API_KEY))

Gemini API key loaded: True
OpenWeather API key loaded: True


In [8]:
client = genai.Client(api_key=GEMINI_API_KEY)

print("Gemini client initialized successfully!")

Gemini client initialized successfully!


## 3. OpenWeather Weather Tool

This function retrieves the current weather for a selected city using the OpenWeather API and returns the city name, temperature in Celsius, and condition description.


In [ ]:
def get_current_weather(city: str) -> dict:
    """
    Retrieve the current weather for a city using OpenWeather.

    Args:
        city: Name of the city.

    Returns:
        A dictionary with the city name, temperature, and weather description.
    """

    url = "https://api.openweathermap.org/data/2.5/weather"

    params = {
        "q": city,
        "appid": OPENWEATHER_API_KEY,
        "units": "metric"
    }

    try:
        response = requests.get(url, params=params, timeout=10)
        response.raise_for_status()

        data = response.json()

        temperature = data["main"]["temp"]
        description = data["weather"][0]["description"]

        result = {
            "city": city,
            "temperature_celsius": temperature,
            "description": description
        }

        print(f"Weather retrieved for {city}: {temperature}°C")
        return result

    except requests.exceptions.RequestException as e:
        return {
            "city": city,
            "error": f"Weather API request failed: {str(e)}"
        }

    except (KeyError, TypeError):
        return {
            "city": city,
            "error": "Unexpected response format from OpenWeather."
        }


### Test the Weather Tool

Before connecting the weather function to Gemini, it’s useful to test it directly with a single city to confirm the OpenWeather integration works correctly.


In [10]:
weather_test = get_current_weather("Kathmandu")

print("\nWeather Tool Result:")
print(weather_test)

Weather retrieved for Kathmandu: 20.12°C

Weather Tool Result:
{'city': 'Kathmandu', 'temperature_celsius': 20.12, 'description': 'few clouds'}


In [ ]:
print(get_current_weather("Kathmandu"))
print(get_current_weather("Notarealplace123"))


Weather retrieved for Kathmandu: 20.12°C
{'city': 'Kathmandu', 'temperature_celsius': 20.12, 'description': 'few clouds'}
{'city': 'Notarealplace123', 'error': 'Weather API request failed: 404 Client Error: Not Found for url: https://api.openweathermap.org/data/2.5/weather?q=Notarealplace123&appid=ba840c790adc9a532f9b65b8be2da782&units=metric'}


## 4. Gemini Weather Agent

The Gemini agent is connected to the weather tool using automatic function calling, so it can decide when it needs live data and call the Python function to fetch it.


In [14]:
weather_agent = client.chats.create(
    model="gemini-3.8-flash",
    config=types.GenerateContentConfig(
        system_instruction=(
            "You are a weather assistant. "
            "Use the get_current_weather tool whenever you need "
            "current weather information. "
            "Return clear and concise weather information."
        ),
        tools=[get_current_weather],
        temperature=0.0
    )
)

print("Gemini Weather Agent created successfully!")

Gemini Weather Agent created successfully!


### Test Gemini Weather Agent

Ask Gemini for the current weather in a single location. It should recognize that it needs live information and call the weather tool automatically.


In [15]:
response = weather_agent.send_message(
    "What is the current weather in Kathmandu?"
)

print("Gemini Response:")
print(response.text)

Weather retrieved for Kathmandu: 20.12°C
Gemini Response:
The current weather in Kathmandu is **20.1°C** with **few clouds**.


## 5. Sequential Weather Retrieval

The weather agent fetches data for the three cities in a fixed order. Once each request is complete, the temperature is kept for the final average calculation.

The locations are processed in this order:

1. Kathmandu
2. London
3. Tokyo


In [ ]:
locations = [
    "Kathmandu",
    "London",
    "Tokyo"
]

weather_results = []

print("Starting sequential weather retrieval...\n")

for city in locations:
    print(f"--- Processing: {city} ---")

    response = weather_agent.send_message(
        f"Use the get_current_weather tool to retrieve the current "
        f"weather for {city}. Return the temperature in Celsius "
        f"and the weather condition."
    )

    print(response.text)
    print()


Starting sequential weather retrieval...

--- Processing: Kathmandu ---
Weather retrieved for Kathmandu: 20.12°C
The current weather in Kathmandu:

- **Temperature:** 20.12°C
- **Weather Condition:** Few clouds

--- Processing: London ---
Weather retrieved for London: 21.55°C
The current weather in London:

- **Temperature:** 21.55°C
- **Weather Condition:** Overcast clouds

--- Processing: Tokyo ---
Weather retrieved for Tokyo: 22.46°C
The current weather in Tokyo:

- **Temperature:** 22.46°C
- **Weather Condition:** Heavy intensity rain



## 6. Calculate Average Temperature

The temperature from each city is collected from the weather tool results, and Python calculates the average in Celsius using the standard formula:

Average Temperature = (Temperature₁ + Temperature₂ + Temperature₃) / 3


In [ ]:
weather_results = []

for city in locations:
    result = get_current_weather(city)
    weather_results.append(result)

print("\n===== Weather Results =====")

for result in weather_results:
    if "temperature_celsius" in result:
        print(
            f"{result['city']}: "
            f"{result['temperature_celsius']}°C - "
            f"{result['description']}"
        )
    else:
        print(
            f"{result['city']}: "
            f"Error - {result.get('error', 'Unknown error')}"
        )

temperatures = [
    result["temperature_celsius"]
    for result in weather_results
    if "temperature_celsius" in result
]

if len(temperatures) == len(locations):
    average_temperature = sum(temperatures) / len(temperatures)

    print("\n===== Average Temperature =====")
    print(f"Average temperature: {average_temperature:.2f}°C")
else:
    print(
        "\nAverage temperature could not be calculated "
        "because one or more weather requests failed."
    )


Weather retrieved for Kathmandu: 19.12°C
Weather retrieved for London: 21.29°C
Weather retrieved for Tokyo: 22.46°C

===== Weather Results =====
Kathmandu: 19.12°C - few clouds
London: 21.29°C - overcast clouds
Tokyo: 22.46°C - heavy intensity rain

===== Average Temperature =====
Average temperature: 20.96°C


## 7. Final Weather Report

The final report brings together all three locations and shows the calculated average temperature. If a request fails, the error is shown without stopping the full workflow.


In [ ]:
print("=" * 50)
print("          FINAL WEATHER REPORT")
print("=" * 50)

successful_temperatures = []

for result in weather_results:
    city = result["city"]

    if "temperature_celsius" in result:
        temperature = result["temperature_celsius"]
        description = result["description"]

        successful_temperatures.append(temperature)

        print(f"\nLocation    : {city}")
        print(f"Temperature : {temperature:.2f}°C")
        print(f"Condition   : {description}")

    else:
        print(f"\nLocation    : {city}")
        print(f"Error       : {result.get('error', 'Unknown error')}")

print("\n" + "-" * 50)

if len(successful_temperatures) == len(locations):
    average_temperature = (
        sum(successful_temperatures) / len(successful_temperatures)
    )

    print(f"Average Temperature: {average_temperature:.2f}°C")
else:
    print("Average temperature could not be calculated.")

print("=" * 50)


          FINAL WEATHER REPORT

Location    : Kathmandu
Temperature : 19.12°C
Condition   : few clouds

Location    : London
Temperature : 21.29°C
Condition   : overcast clouds

Location    : Tokyo
Temperature : 22.46°C
Condition   : heavy intensity rain

--------------------------------------------------
Average Temperature: 20.96°C


## 8. Conclusion

This notebook shows how a Gemini-powered agent can use automatic function calling to retrieve live weather data from the OpenWeather API for several cities, process the results sequentially, and calculate the average temperature using Python instead of relying on the model to do arithmetic. It also includes simple error handling for invalid cities, missing keys, and network issues.
